Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

from typing import Type
from sklearn.base import BaseEstimator

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
)

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'opt-model'
add_smote = False
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 5000 # 1000

save_model_and_metrics = False

## Optimization functions

In [5]:
def optimize_estimator_optuna(
    target:str,
    estimator:Type[BaseEstimator],
    estimator_params:dict,
    objective:callable,
    n_trials:int,
    step_scoring_average:str=step_scoring_average,
    params_to_process:list=None,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
):
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
    )
    
    opt = OptunaOptimizer(
        objective=partial(objective, ml_pipe=ml_pipe),
        study_name="model_study",
        direction="maximize",
    )
    
    opt.optimize(n_trials=n_trials)
    
    best_params = opt.study.best_params
    
    if params_to_process:
        for param in params_to_process:
            # Find one key in best_params which contains param
            key = next((k for k in best_params.keys() if param in k), None)
            if key:
                best_params[param] = best_params.pop(key)

    print('best_params')
    display(best_params)
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params={**estimator_params, **best_params},
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        step_scoring_average=step_scoring_average, # No need to pass, it would not be used
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# Logistic Regression

In [6]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}
params_to_process = ['penalty']

def logreg_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
    
    # # NOTE: Does not work, since optuna requires static parameters ranges
    # solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'])
    # penalty_options = {
    #     'lbfgs': ['l2'],
    #     'liblinear': ['l1', 'l2'],
    #     'newton-cg': ['l2'],
    #     'newton-cholesky': ['l2'],
    #     'sag': ['l2'],
    #     'saga': ['l1', 'l2', 'elasticnet'],
    # }
    # penalty = trial.suggest_categorical('penalty', penalty_options[solver])
    # NOTE: warnings + inconvenient to create best model
    # solver_penalty_combinations = [
    #     ("liblinear", "l1"),
    #     ("liblinear", "l2"),
    #     ("saga", "l1"),
    #     ("saga", "l2"),
    #     ("saga", "elasticnet"),
    #     ("lbfgs", "l2"),
    #     ("newton-cg", "l2"),
    #     ("newton-cholesky", "l2"),
    #     ("sag", "l2"),
    # ]
    # solver, penalty = trial.suggest_categorical(
    #     'solver_penalty', solver_penalty_combinations
    # )
    
    solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'])
    # solver = trial.suggest_categorical('solver', ['lbfgs',])
    
    if solver == 'liblinear':
        penalty = trial.suggest_categorical('penalty_liblinear', ['l1', 'l2'])
    elif solver == 'saga':
        penalty = trial.suggest_categorical('penalty_saga', ['l1', 'l2', 'elasticnet'])
    else:
        penalty = trial.suggest_categorical('penalty_other', ['l2'])
    
    # If C is large, the penalty should be small
    C = trial.suggest_loguniform('C', 1e-4, 1e3) 
    
    l1_ratio = None
    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_uniform('l1_ratio', 0., 1.)
    
    class_weight = trial.suggest_categorical(
        'class_weight', [None, 'balanced']
    )
    
    suggested_estimator_params = {
        "solver": solver,
        "penalty": penalty,
        "C": C,
        "l1_ratio": l1_ratio,
        "class_weight": class_weight,
    }
    
    
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=logreg_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-14 18:56:58,502] A new study created in memory with name: model_study
[I 2025-04-14 18:56:58,583] Trial 0 finished with value: 0.7985639090256361 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l2', 'C': 1.6136341713591311, 'class_weight': None}. Best is trial 0 with value: 0.7985639090256361.
[I 2025-04-14 18:56:58,644] Trial 1 finished with value: 0.7941608217143109 and parameters: {'solver': 'lbfgs', 'penalty_other': 'l2', 'C': 0.47129737561107793, 'class_weight': None}. Best is trial 0 with value: 0.7985639090256361.
[I 2025-04-14 18:56:58,701] Trial 2 finished with value: 0.26121337827326935 and parameters: {'solver': 'saga', 'penalty_saga': 'elasticnet', 'C': 0.00021142332035497166, 'l1_ratio': 0.6075448519014384, 'class_weight': None}. Best is trial 0 with value: 0.7985639090256361.
[I 2025-04-14 18:56:58,755] Trial 3 finished with value: 0.797286337918985 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l1', 'C': 0.292575779498244, 'class_w

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_opt-model
holdout_test_f1_macro,0.772036
holdout_test_accuracy_balanced,0.77662
holdout_test_roc_auc,0.886574
holdout_test_f1,0.829787
holdout_test_accuracy,0.786667
cv_test_f1_macro_median,0.802827
cv_test_accuracy_balanced_median,0.818111
cv_test_roc_auc_median,0.871517


In [7]:
estimator = LogisticRegression
target = 'no_fragmentation'
estimator_params = {
    "fit_intercept": False,
}
params_to_process = ['penalty']

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=logreg_objective,
    n_trials=n_trials,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-14 19:36:35,323] A new study created in memory with name: model_study
[I 2025-04-14 19:36:35,369] Trial 0 finished with value: 0.8397856760138982 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l2', 'C': 1.6136341713591311, 'class_weight': None}. Best is trial 0 with value: 0.8397856760138982.
[I 2025-04-14 19:36:35,415] Trial 1 finished with value: 0.8027588628638668 and parameters: {'solver': 'lbfgs', 'penalty_other': 'l2', 'C': 0.47129737561107793, 'class_weight': None}. Best is trial 0 with value: 0.8397856760138982.
[I 2025-04-14 19:36:35,456] Trial 2 finished with value: 0.4233049487844008 and parameters: {'solver': 'saga', 'penalty_saga': 'elasticnet', 'C': 0.00021142332035497166, 'l1_ratio': 0.6075448519014384, 'class_weight': None}. Best is trial 0 with value: 0.8397856760138982.
[I 2025-04-14 19:36:35,498] Trial 3 finished with value: 0.7911634439518859 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l1', 'C': 0.292575779498244, 'class_w

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LogisticRegression"


,0
target,no_fragmentation
model,LogisticRegression_no_fragmentation_opt-model
holdout_test_f1_macro,0.798595
holdout_test_accuracy_balanced,0.834091
holdout_test_roc_auc,0.943636
holdout_test_f1,0.723404
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.865709
cv_test_accuracy_balanced_median,0.882051
cv_test_roc_auc_median,0.938462
